# Flow Reversal Steering による死角物体の探索・把持

π0.5（学習済み）を追加学習なしで使い、人間の大まかな方向の指示を FRS で π0.5 の動きに変換して誘導する。ランタイムのタイプで GPU（T4）を選び、上から順に実行する（アカウントや認証情報は不要）。

## セル0: セットアップ（ライブラリのインストール・環境変数・トークナイザの取得）

In [ ]:
# セル0: セットアップ（ライブラリのインストール・環境変数・トークナイザの取得）
import importlib
import importlib.util
import os
import pathlib
import subprocess
import sys

# MuJoCo を画面なしで描画する
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("MUJOCO_EGL_DEVICE_ID", "0")

# Colab の GPU ランタイムは NVIDIA の EGL を glvnd に登録していないため、MUJOCO_GL=egl でも
# CPU のソフトウェア描画（Mesa llvmpipe）に落ちる。ドライバと同じ版の libEGL_nvidia は
# /usr/lib64-nvidia にあるので、それを登録して GPU 描画にする（版が違う libEGL_nvidia を apt で入れると
# EGL デバイスが 0 個になり描画できない。2026-10-24 に確認）。
# （プロセス内で最初に描画する前に実行する必要がある）
EGL_VENDOR = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
driver = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()
egl_lib = next(
    (str(p) for d in ["/usr/lib64-nvidia", "/usr/lib/x86_64-linux-gnu"] for p in pathlib.Path(d).glob(f"libEGL_nvidia.so.{driver}")),
    None,
)
if egl_lib:
    EGL_VENDOR.write_text('{"file_format_version": "1.0.0", "ICD": {"library_path": "%s"}}\n' % egl_lib)
print(f"NVIDIA driver {driver} / EGL: {egl_lib or '見つからない（CPU 描画のまま。動作はするが遅い）'}")

LEROBOT_REV = "d3793b7e36522321c1c455f0dc4826db3793dceb"  # LeRobot main (2026-10-23)
if importlib.util.find_spec("lerobot") is None:
    # LIBERO が依存する egl_probe は、pip の隔離ビルド環境の cmake ではビルドに失敗するため、
    # システムの cmake を使って先に入れておく（Colab, Python 3.13 で確認）
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "hf-egl-probe", "egl_probe"],
        check=True,
    )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            f"lerobot[libero,pi] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_REV}",
            "imageio[ffmpeg]",
            # Colab 既定の torchao 0.10 では、LeRobot が読み込む diffusers の import が失敗する
            "torchao>=0.18",
        ],
        check=True,
    )
    importlib.invalidate_caches()

# LIBERO は初回 import 時に input() で保存先を尋ねてくるので、既定値の config.yaml を先に書いておく
libero_cfg = pathlib.Path(os.environ.get("LIBERO_CONFIG_PATH", "~/.libero")).expanduser() / "config.yaml"
if not libero_cfg.exists():
    import yaml

    root = pathlib.Path(importlib.util.find_spec("libero.libero").origin).parent
    libero_cfg.parent.mkdir(parents=True, exist_ok=True)
    libero_cfg.write_text(
        yaml.safe_dump(
            {
                "benchmark_root": str(root),
                "bddl_files": str(root / "bddl_files"),
                "init_states": str(root / "init_files"),
                "datasets": str(root.parent / "datasets"),
                "assets": str(root / "assets"),
            }
        )
    )

# π0.5 のトークナイザ（google/paligemma-3b-pt-224）は Hugging Face では gated（ログインとライセンス同意が必要）。
# 認証なしで実行できるよう、このリポジトリに同梱したもの（Gemma 利用規約に従い無改変で再配布）を取得して使う。
REPO_RAW = "https://raw.githubusercontent.com/pai-adv-public/pai-final-2026/main"
TOKENIZER_DIR = pathlib.Path("/content/paligemma_tokenizer")
TOKENIZER_FILES = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "tokenizer.model"]
if not all((TOKENIZER_DIR / f).exists() for f in TOKENIZER_FILES):
    import urllib.request

    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    for f in TOKENIZER_FILES:
        urllib.request.urlretrieve(f"{REPO_RAW}/assets/paligemma_tokenizer/{f}", TOKENIZER_DIR / f)

OUT = pathlib.Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
print("setup ok")

## セル1: π0.5（学習済み）と LIBERO シミュレータの読み込み

In [ ]:
# セル1: π0.5（学習済み）と LIBERO シミュレータの読み込み
import time

import numpy as np
import torch

from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import LiberoEnv as LiberoEnvConfig
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.envs.utils import preprocess_observation
from lerobot.policies.factory import make_policy, make_pre_post_processors

POLICY_ID = "lerobot/pi05_libero_finetuned"
SUITE, TASK_ID = "libero_object", 0
N_ACTION_STEPS = 10  # 1 回の推論で実行するステップ数（openpi と同じ）
DEVICE = "cuda"

# --- シミュレータ（LeRobot の評価と同じ設定: 固定カメラ + 手先カメラ, 360x360）---
env_cfg = LiberoEnvConfig(task=SUITE, task_ids=[TASK_ID])
vec_env = make_env(env_cfg, n_envs=1)[SUITE][TASK_ID]  # SyncVectorEnv（バッチ次元 1）
env = vec_env.envs[0]  # lerobot.envs.libero.LiberoEnv
print("task:", env.task_description, "| max steps:", env._max_episode_steps)

# --- 方策 ---
t0 = time.time()
policy_cfg = PreTrainedConfig.from_pretrained(POLICY_ID)
policy_cfg.pretrained_path = POLICY_ID
policy_cfg.n_action_steps = N_ACTION_STEPS
policy_cfg.device = DEVICE
policy_cfg.compile_model = False  # Hub の設定では有効だが、初回の自動チューニングに約 8 分かかるため無効にする
assert not policy_cfg.use_relative_actions and not policy_cfg.use_visual_memory
# T4（メインメモリ 12 GB・VRAM 15 GB）でも読み込めるよう、モデルを最初から GPU 上に bf16 で作る。
# （LeRobot の既定では CPU 上に fp32 で作ってから bf16 に変換するため、約 15 GB のメインメモリが要る）
torch.set_default_dtype(torch.bfloat16)
try:
    with torch.device(DEVICE):
        policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg)
finally:
    torch.set_default_dtype(torch.float32)
# 上の作り方だと、本来 fp32 の行動側の層（action_in_proj など）まで bf16 になるので、チェックポイントから fp32 で読み直す
from huggingface_hub import hf_hub_download
from safetensors import safe_open

_params = dict(policy.named_parameters())
with safe_open(hf_hub_download(POLICY_ID, "model.safetensors"), framework="pt") as f:
    for key in f.keys():
        if "paligemma_with_expert" not in key and key in _params:
            _params[key].data = f.get_tensor(key).to(DEVICE)
assert all(p.dtype == torch.float32 for k, p in _params.items() if "paligemma_with_expert" not in k)
policy.eval()
pre, post = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=POLICY_ID,
    preprocessor_overrides={
        "device_processor": {"device": DEVICE},
        "rename_observations_processor": {"rename_map": {}},
        "tokenizer_processor": {"tokenizer_name": str(TOKENIZER_DIR)},  # 同梱のトークナイザ（認証不要）
    },
)
env_pre, _ = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)
print(
    f"policy loaded in {time.time() - t0:.0f}s | dtype={policy_cfg.dtype} "
    f"| chunk={policy_cfg.chunk_size} | steps={policy_cfg.num_inference_steps} "
    f"| GPU mem {torch.cuda.memory_allocated() / 2**30:.1f} GiB"
)


def to_batch(obs):
    """vec_env の観測を π0.5 の入力バッチに変換する（lerobot-eval と同じ順序）。"""
    batch = preprocess_observation(obs)
    batch["task"] = [env.task_description]
    batch = env_pre(batch)
    return pre(batch)


def frame_of(obs):
    """動画用: 固定カメラと手先カメラを横に並べる。
    robosuite の画像は上下が逆なので上下だけ戻す（π0.5 への入力は env_pre が学習時と同じ 180° 回転にする。表示とは別）。"""
    agent = obs["pixels"]["image"][0][::-1]
    wrist = obs["pixels"]["image2"][0][::-1]
    return np.concatenate([agent, wrist], axis=1)


@torch.no_grad()
def pi05_chunk(batch, noise=None):
    """通常の π0.5: 正規化された行動空間で (1, N_ACTION_STEPS, 7) を返す。"""
    return policy.predict_action_chunk(batch, noise=noise)[:, :N_ACTION_STEPS]


def run_episode(chunk_fn, init_state=0, seed=0, max_steps=None):
    """chunk_fn(batch, obs) -> 正規化行動チャンク を繰り返し実行する。"""
    max_steps = max_steps or env._max_episode_steps
    env.init_state_id = init_state
    obs, _ = vec_env.reset(seed=seed)
    frames, success, steps, infer_sec = [frame_of(obs)], False, 0, []
    while steps < max_steps and not success:
        batch = to_batch(obs)
        t = time.time()
        chunk = chunk_fn(batch, obs)
        infer_sec.append(time.time() - t)
        for i in range(chunk.shape[1]):
            action = post(chunk[:, i]).cpu().numpy()
            obs, _, terminated, _, _ = vec_env.step(action)
            frames.append(frame_of(obs))
            steps += 1
            success = bool(env._env.check_success())
            if success or terminated[0] or steps >= max_steps:
                break
    return {"success": success, "steps": steps, "frames": frames, "infer_sec": infer_sec}


def show_video(frames, name, fps=20):
    """フレーム列を MP4 に保存し、Notebook 内で再生する。"""
    import imageio.v2 as imageio
    from IPython.display import Video, display

    path = OUT / name
    imageio.mimsave(path, frames, fps=fps, macro_block_size=8)
    display(Video(str(path), embed=True, width=720))

## セル2: 動作確認（元の配置で、通常の π0.5 がスープ缶を掴んでかごに入れられるか）

In [ ]:
# セル2: 動作確認（元の配置で、通常の π0.5 がスープ缶を掴んでかごに入れられるか）
res = run_episode(lambda batch, obs: pi05_chunk(batch), init_state=0, seed=0)
print(
    f"success={res['success']} steps={res['steps']} "
    f"推論時間: 初回 {res['infer_sec'][0]:.2f}s / 2 回目以降の平均 {np.mean(res['infer_sec'][1:]):.3f}s"
)
show_video(res["frames"], "smoke.mp4")  # 左: 固定カメラ / 右: 手先カメラ

## セル3: Flow Reversal Steering (FRS) の実装

In [ ]:
# セル3: Flow Reversal Steering (FRS) の実装
# 参考: Tang et al., "Improving Robotic Generalist Policies via Flow Reversal Steering" (arXiv 2606.13675)
#   1. 粗い参照行動 a_ref（例: 「左へ動け」）を作る
#   2. 観測で条件づけた流れの ODE を逆向きに積分し、a_ref に対応するノイズ â0 を求める
#   3. â0 から通常どおり順向きに積分すると、a_ref に近い「π0.5 らしい」行動が得られる
# π0.5 (openpi / LeRobot) は t=1 がノイズ、t=0 が行動の規約（NOISE_AT_ONE）。
from lerobot.policies.common.vla_utils import make_att_2d_masks
from lerobot.processor.normalize_processor import NormalizerProcessorStep
from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS

FLOW_STEPS = policy_cfg.num_inference_steps  # 10（論文と同じ）
CHUNK = policy_cfg.chunk_size  # 50
PAD_DIM = policy_cfg.max_action_dim  # 32（π0.5 内部では行動を 32 次元にゼロ埋め）
ACT_DIM = 7  # LIBERO: 手先の相対位置 3 + 相対回転 3 + グリッパー 1
# 参照行動の大きさ（LIBERO の行動空間 [-1, 1] での値）と、チャンクの先頭何ステップを参照で固定するか。
# 物体が見えている状態で「スープへ向かう」π0.5 を逆向きに誘導できるかで決めた（セル4 の後の比較）:
#   0.5 / 10 ステップ（論文と同じ方式）では逆向き（ロボット基準の +y）へ動かせず、1.0 / チャンク全体で +0.23 m 動いた。
REF_SPEED = 1.0
REF_STEPS = CHUNK

# 指示 -> ロボット基準座標での移動方向。名前は固定カメラの画面上の向きに合わせる
# （ロボット基準の +y は画面の右。カメラはロボットの正面からロボットを向いている）
DIRECTIONS = {
    "right": np.array([0.0, 1.0, 0.0]),
    "left": np.array([0.0, -1.0, 0.0]),
    "forward": np.array([1.0, 0.0, 0.0]),  # 手前（カメラ側）
    "back": np.array([-1.0, 0.0, 0.0]),  # 奥（ロボット側）
}
GRIPPER_OPEN = -1.0

normalizer = next(s for s in pre.steps if isinstance(s, NormalizerProcessorStep))


@torch.no_grad()
def velocity_field(batch):
    """観測で条件づけた速度場 v(x, t) を返す。画像・言語部分の KV キャッシュは 1 回だけ計算する。

    LeRobot の PI05Pytorch.sample_actions の前半と同じ処理。
    """
    m = policy.model
    images, img_masks = policy._preprocess_images(batch)
    states, state_masks = policy._prepare_memory_states(batch)
    embs, pad, att = m.embed_prefix(
        images, img_masks, batch[OBS_LANGUAGE_TOKENS], batch[OBS_LANGUAGE_ATTENTION_MASK], states, state_masks
    )
    att_4d = m._prepare_attention_masks_4d(
        make_att_2d_masks(pad, att), dtype=embs.dtype if m.use_typed_attention_masks else None
    )
    m.paligemma_with_expert.paligemma.model.language_model.config._attn_implementation = "eager"
    _, kv = m.paligemma_with_expert.forward(
        attention_mask=att_4d,
        position_ids=torch.cumsum(pad, dim=1) - 1,
        past_key_values=None,
        inputs_embeds=[embs, None],
        use_cache=True,
    )

    def v(x, t):
        time = torch.full((x.shape[0],), t, dtype=torch.float32, device=x.device)
        return m.denoise_step(prefix_pad_masks=pad, past_key_values=kv, x_t=x, timestep=time)

    return v


def integrate_forward(v, noise):
    """ノイズ -> 行動（t: 1 -> 0）。LeRobot の euler_integrate と同じ更新式。"""
    x, dt = noise, -1.0 / FLOW_STEPS
    for k in range(FLOW_STEPS):
        x = x + dt * v(x, 1.0 + k * dt)
    return x


def integrate_reverse(v, action):
    """行動 -> ノイズ（t: 0 -> 1）。順方向のオイラー法を 1 ステップずつ逆にたどる。"""
    x, h = action, 1.0 / FLOW_STEPS
    for k in range(FLOW_STEPS):
        x = x + h * v(x, k * h)
    return x


def make_reference(command, speed=None):
    """指示から参照行動チャンク (1, CHUNK, PAD_DIM) を作る（正規化された行動空間）。"""
    speed = REF_SPEED if speed is None else speed
    raw = torch.zeros(1, CHUNK, ACT_DIM)
    raw[..., :3] = torch.as_tensor(DIRECTIONS[command] * speed, dtype=torch.float32)
    raw[..., 6] = GRIPPER_OPEN
    ref = torch.zeros(1, CHUNK, PAD_DIM, device=DEVICE)
    ref[..., :ACT_DIM] = normalizer._normalize_action(raw, inverse=False).to(DEVICE)
    return ref


@torch.no_grad()
def frs_chunk(batch, command, ref_steps=None, return_noise=False):
    """FRS: 参照行動を逆積分してノイズにし、順積分で π0.5 の行動に戻す。

    論文と同様、先頭 ref_steps 以外のノイズは N(0, I) に置き換え、残りは π0.5 に補完させる。
    return_noise=True なら、π0.5 に入力したノイズも返す（可視化用）。
    """
    ref_steps = REF_STEPS if ref_steps is None else ref_steps
    v = velocity_field(batch)
    noise = integrate_reverse(v, make_reference(command))
    noise[:, ref_steps:] = torch.randn_like(noise[:, ref_steps:])
    actions = integrate_forward(v, noise)[:, :N_ACTION_STEPS, :ACT_DIM]
    return (actions, noise) if return_noise else actions


## セル4: FRS の単体確認

In [ ]:
# セル4: FRS の単体確認
#   (1) 自前の順積分が LeRobot の π0.5 と同じ行動を出すか
#   (2) 逆積分 -> 順積分の往復でどれだけ元に戻るか
#   (3) 「左」「右」の指示で手先が実際にその方向へ動くか（誘導なしの π0.5 と比較）
env.init_state_id = 0
obs, _ = vec_env.reset(seed=0)
batch = to_batch(obs)
torch.manual_seed(0)
noise = torch.randn(1, CHUNK, PAD_DIM, device=DEVICE)
v = velocity_field(batch)
a_mine = integrate_forward(v, noise)
a_lib = policy.predict_action_chunk(batch, noise=noise.clone())
print(f"(1) 順積分の一致: max|自前 - LeRobot| = {(a_mine[..., :ACT_DIM] - a_lib).abs().max().item():.4f}")
noise_back = integrate_reverse(v, a_mine)
a_again = integrate_forward(v, noise_back)
print(
    f"(2) 往復: mean|noise - 逆積分| = {(noise_back - noise)[..., :ACT_DIM].abs().mean().item():.4f}, "
    f"mean|a - 往復後の a| = {(a_again - a_mine)[..., :ACT_DIM].abs().mean().item():.4f}"
)


def eef_pos(obs):
    return obs["robot_state"]["eef"]["pos"][0].copy()


def run_command(mode, command, n_chunks=3, init_state=0, seed=0):
    """mode: "frs" | "pi05"。n_chunks 回（各 N_ACTION_STEPS ステップ）実行し、手先の移動量を返す。"""
    torch.manual_seed(seed)
    env.init_state_id = init_state
    obs, _ = vec_env.reset(seed=seed)
    start, frames = eef_pos(obs), [frame_of(obs)]
    for _ in range(n_chunks):
        batch = to_batch(obs)
        if mode == "frs":
            chunk = frs_chunk(batch, command)
        else:
            chunk = pi05_chunk(batch)
        for i in range(chunk.shape[1]):
            obs, *_ = vec_env.step(post(chunk[:, i]).cpu().numpy())
            frames.append(frame_of(obs))
    return eef_pos(obs) - start, frames


print("(3) 30 ステップ後の手先の移動量 [m]（ロボット基準座標 x, y, z）")
clips = []
for mode, command in [("pi05", None), ("frs", "left"), ("frs", "right")]:
    delta, frames = run_command(mode, command)
    print(f"  {mode:6s} {str(command):5s} dx={delta[0]:+.3f} dy={delta[1]:+.3f} dz={delta[2]:+.3f}")
    clips += frames
show_video(clips, "frs_check.mp4")  # 誘導なし → FRS で左 → FRS で右 の順

## セル5: 対象物（スープ缶）をかごの陰に置く — 固定カメラからはかごに隠れ、手先カメラにも映らない位置

In [ ]:
# セル5: 対象物（スープ缶）をかごの陰に置く — 固定カメラからはかごに隠れ、手先カメラにも映らない位置
import imageio.v2 as imageio
from IPython.display import Image, display

TARGET = "alphabet_soup_1"
HIDE_XY = (-0.15, 0.30)  # かごの裏（ロボット側）。候補を画像で見比べ、把持まで通る位置を選んだ
IDLE = np.array([[0, 0, 0, 0, 0, 0, -1.0]])  # 何もしない行動（グリッパーは開）


def reset_hidden(seed=0):
    """元の初期状態から、対象物だけをかごの陰に移して落ち着かせる。"""
    env.init_state_id = 0
    vec_env.reset(seed=seed)
    sim = env._env.sim  # リセットのたびに作り直されるので毎回取り直す
    joint = env._env.env.objects_dict[TARGET].joints[0]
    qpos = sim.data.get_joint_qpos(joint).copy()
    qpos[:2] = HIDE_XY
    sim.data.set_joint_qpos(joint, qpos)
    sim.forward()
    for _ in range(10):
        obs, *_ = vec_env.step(IDLE)
    return obs


obs = reset_hidden()
imageio.imwrite(OUT / "scene_hidden.png", frame_of(obs))
display(Image(str(OUT / "scene_hidden.png")))  # 左: 固定カメラ / 右: 手先カメラ。どちらにもスープ缶は映らない

## セル6: FRS による探索 → π0.5 への引き渡し

In [ ]:
# セル6: FRS による探索 → π0.5 への引き渡し
# 最初は π0.5 に任せて様子を見る（auto）。うまくいかなそうなら人間が right / left / forward / back で介入し（FRS）、
# 手先カメラにスープ缶が見えたら handover で π0.5 に任せる（以降の行動はすべて π0.5 が決める。移動の途中で出してもよい）。
#   INTERACTIVE = True  : 1 指示ごとに入力を受け付ける（1 指示 = 10 ステップ。空入力は auto）
#   INTERACTIVE = False : COMMANDS を順に再生する（上から一括実行用）
INTERACTIVE = False
# 様子見（π0.5 が中央の別の物体へ向かい始める）-> 介入して右（かごの方）へ探す -> 奥に寄せて缶を中央に -> 任せる
COMMANDS = ["auto"] * 2 + ["right"] * 5 + ["back"] * 2 + ["handover"]
MAX_STEPS = 400


def run_search(commands=(), steer=True, interactive=False, seed=0):
    """steer=True: 指示を FRS で π0.5 に反映して探索し、handover 以降は π0.5 に任せる。
    指示 auto はその 10 ステップを π0.5 に任せる（様子見）。steer=False: 最初から最後まで π0.5 のみ（比較用）。"""
    torch.manual_seed(seed)
    obs = reset_hidden(seed)
    queue, mode, steps, success = list(commands), ("operator" if steer else "pi05"), 0, False
    frames = [frame_of(obs)]
    while steps < MAX_STEPS and not success:
        cmd = None
        if mode == "operator":  # 人間の指示を受け付ける段階（handover まで）
            if interactive:
                imageio.imwrite(OUT / "now.png", frame_of(obs))
                display(Image(str(OUT / "now.png")))
                cmd = input("auto(Enter) / right / left / forward / back / handover > ").strip() or "auto"
            else:
                cmd = queue.pop(0) if queue else "handover"
            if cmd == "handover":
                mode = "pi05"
            elif cmd != "auto" and cmd not in DIRECTIONS:
                print("auto / right / left / forward / back / handover のいずれかを入力")
                continue
        batch = to_batch(obs)
        steering = mode == "operator" and cmd in DIRECTIONS
        chunk = frs_chunk(batch, cmd) if steering else pi05_chunk(batch)
        for i in range(chunk.shape[1]):
            obs, _, terminated, _, _ = vec_env.step(post(chunk[:, i]).cpu().numpy())
            frames.append(frame_of(obs))
            steps += 1
            success = bool(env._env.check_success())
            if success or terminated[0] or steps >= MAX_STEPS:
                break
    print(f"{'FRS で探索 → π0.5 に引き渡し' if steer else 'π0.5 のみ'}: success={success} steps={steps}")
    return {"success": success, "steps": steps, "frames": frames}


res_frs = run_search(COMMANDS, steer=True, interactive=INTERACTIVE)
show_video(res_frs["frames"], "search_frs.mp4")

## セル7: 比較 — 人間が手伝わず、π0.5 に指示文だけを与えた場合（缶が見えないので別の物体を掴んでしまう）

In [ ]:
# セル7: 比較 — 人間が手伝わず、π0.5 に指示文だけを与えた場合（缶が見えないので別の物体を掴んでしまう）
res_pi05 = run_search(steer=False)
show_video(res_pi05["frames"], "pi05_only.mp4")